# multimodal-query-demo — one-click reproduce

This notebook clones the repository, installs a pinned Rust toolchain, runs the full test suite, regenerates the benchmark report, renders the five figures, and (optionally) rebuilds the paper PDF.

**Expected runtime:** ~8-12 minutes on a Colab CPU runtime. The paper rebuild (`texlive-latex-extra`) is the single largest cost and is guarded by a flag.

**What you get at the end:** `out/bench.json`, `paper/figures/*.png` (five PNGs), and optionally `paper/multimodal-query-demo.pdf`, all reproducible bit-for-bit from a clean VM given the same toolchain version.

**What this notebook does not claim:** that the numbers generalise to any other machine. Bench latencies are single-host wall-clock, and absolute values will differ on Colab CPU vs. the author's workstation. The *shapes* (ratios, slopes, pruning counts) are what reproduce.

## 1. Clone the repository

In [ ]:
%%bash
set -eu
if [ ! -d multimodal-query-demo ]; then
  git clone --depth 1 https://github.com/infinityabundance/multimodal-query-demo.git
fi
cd multimodal-query-demo && git log -1 --oneline

## 2. Install Rust (pinned to workspace MSRV)

The install is idempotent: if `cargo` is already on PATH (local run, previous Colab cell), the download is skipped.

In [ ]:
%%bash
set -eu
if [ ! -x "$HOME/.cargo/bin/cargo" ] && ! command -v cargo >/dev/null; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.85.0 --profile minimal
fi
export PATH="$HOME/.cargo/bin:$PATH"
cargo --version && rustc --version

## 3. Run the workspace test suite

39+ tests including `non_claim_lock` (parses the paper's non-claims against the Rust constant), `deterministic_replay` (SHA-256 fingerprint of a seeded proximity output), and `concurrent_snapshots` (reader/writer stress). Green here means the artefact is structurally intact on this machine.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
cargo test --workspace --release

## 4. Generate a fixture, ingest it, and run every engine path

Hits every `mqd` subcommand (`generate`, `ingest`, `query latest-at`, `query range`, `op proximity`, `op speed`, `non-claims`). `--explain` on the queries dumps the `QueryPlan` (partition counts, pruning counts, scanned rows) so the engine's work is visible, not just the wall clock.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
mkdir -p data out paper/figures
cargo run -p mqd-cli --release -- generate --seed 42 --agents 8 --ticks 10000 --out data/run.jsonl
cargo run -p mqd-cli --release -- ingest --input data/run.jsonl --batch-rows 4096
cargo run -p mqd-cli --release -- query latest-at --input data/run.jsonl --at 25000000000 --explain
cargo run -p mqd-cli --release -- query range --input data/run.jsonl --start 0 --end 10000000000 --entity /agent/0 --kind transform3d --explain
cargo run -p mqd-cli --release -- op proximity --input data/run.jsonl --at 25000000000 --radius 2.5 --json | head -20
cargo run -p mqd-cli --release -- op speed --input data/run.jsonl --entity /agent/0 --json | head -20
cargo run -p mqd-cli --release -- non-claims

## 5. Bench (51 samples) and render figures

`bench` produces a single JSON file with p50/p95/p99 per suite × scale. `figs` reads that JSON and writes five PNGs: `ingest_latency.png`, `latest_at_latency.png`, `range_scan_latency.png`, `proximity_latency.png`, `speed_latency.png`. Each carries a p50 solid line, p95 dashed line, and (for proximity / speed) a reference-complexity curve.

In [ ]:
%%bash
set -eu
export PATH="$HOME/.cargo/bin:$PATH"
cd multimodal-query-demo
cargo run -p mqd-cli --release -- bench --out out/bench.json --samples 51
cargo run -p mqd-cli --release -- figs --from out/bench.json --out paper/figures/
ls -la paper/figures/

## 6. Inspect the bench report and figures

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('multimodal-query-demo/out/bench.json').read_text())
print(f"version={report['version']}  machine={report['machine']}  runs={len(report['runs'])}")
for r in report['runs']:
    print(f"  {r['suite']:12s} scale={r['scale']:10s} x={r['x']:>8.1f}  p50={r['p50_us']:>9.1f}µs  p95={r['p95_us']:>9.1f}µs  p99={r['p99_us']:>9.1f}µs  returned={r['returned_rows']}")

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for png in sorted(Path('multimodal-query-demo/paper/figures').glob('*_latency.png')):
    print(png.name)
    display(Image(filename=str(png)))

## 7. (Optional) Rebuild the paper PDF

Adds ~5 minutes on a cold Colab due to the `texlive-latex-extra` install. Skip it if you only care about the code and figures.

In [ ]:
BUILD_PDF = False  # flip to True to install LaTeX and rebuild the paper

In [ ]:
%%bash -s "$BUILD_PDF"
set -eu
if [ "$1" != "True" ]; then echo 'skipped (set BUILD_PDF = True to enable)'; exit 0; fi
apt-get -qq update && apt-get -qq install -y texlive-latex-extra latexmk >/dev/null
cd multimodal-query-demo && ./paper/build.sh
ls -la paper/multimodal-query-demo.pdf

## Done

What you should have:

- `multimodal-query-demo/out/bench.json` — single-file bench report (p50/p95/p99 per suite × scale).
- `multimodal-query-demo/paper/figures/*.png` — five figures referenced by the paper.
- Optional: `multimodal-query-demo/paper/multimodal-query-demo.pdf` — the 9-page paper with the numbered non-claims charter.

If any cell in §3 (tests) failed, the artefact is broken on this runtime; please report the failing test name.